# noise_reachability step 17: first-inference features of the step-16 map states (Kaggle, 2 x T4)

Step 17 of `noise_reachability/README.md`. No training and no new map: one episode of the base with `repeat` id
10000 on each of the 120 map states (and id 10001 on 4 of them), which stores SmolVLA's prefix
features at the episode's first inference. Each episode must reproduce step 16's committed row: at least 114 of
120 states, and the duplicate id must give the same features bit for bit; otherwise the run stops and is not read.

**Needs no input. Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing. One session
(about 1.5 h, an estimate). If step 10 says work remains, add this version's output as input (**Add Input > Your
Work**, only the latest step 17 version) and Save & Run All again; step 6 must then print `RESUMING`.

Download from the Output tab: `step17_results.zip`.

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "b80c933388684cd11f0d3283fd2e704a8b98b80a"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 17; nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [0, 4, 6, 7]                 # the wrapper check runs on each
N_ACTION_STEPS = 10
MAP_STATES = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
IDS = {"0/0": "10000-10001", "0/1": "10000", "0/2": "10000", "0/3": "10000", "0/4": "10000", "0/5": "10000", "0/6": "10000", "0/7": "10000", "0/8": "10000", "0/9": "10000", "0/10": "10000", "0/11": "10000", "0/12": "10000", "0/13": "10000", "0/14": "10000", "0/15": "10000", "0/16": "10000", "0/17": "10000", "0/18": "10000", "0/19": "10000", "0/20": "10000", "0/21": "10000", "0/22": "10000", "0/23": "10000", "0/24": "10000", "0/25": "10000", "0/26": "10000", "0/27": "10000", "0/28": "10000", "0/29": "10000", "4/0": "10000-10001", "4/1": "10000", "4/2": "10000", "4/3": "10000", "4/4": "10000", "4/5": "10000", "4/6": "10000", "4/7": "10000", "4/8": "10000", "4/9": "10000", "4/10": "10000", "4/11": "10000", "4/12": "10000", "4/13": "10000", "4/14": "10000", "4/15": "10000", "4/16": "10000", "4/17": "10000", "4/18": "10000", "4/19": "10000", "4/20": "10000", "4/21": "10000", "4/22": "10000", "4/23": "10000", "4/24": "10000", "4/25": "10000", "4/26": "10000", "4/27": "10000", "4/28": "10000", "4/29": "10000", "6/0": "10000-10001", "6/1": "10000", "6/2": "10000", "6/3": "10000", "6/4": "10000", "6/5": "10000", "6/6": "10000", "6/7": "10000", "6/8": "10000", "6/9": "10000", "6/10": "10000", "6/11": "10000", "6/12": "10000", "6/13": "10000", "6/14": "10000", "6/15": "10000", "6/16": "10000", "6/17": "10000", "6/18": "10000", "6/19": "10000", "6/20": "10000", "6/21": "10000", "6/22": "10000", "6/23": "10000", "6/24": "10000", "6/25": "10000", "6/26": "10000", "6/27": "10000", "6/28": "10000", "6/29": "10000", "7/0": "10000-10001", "7/1": "10000", "7/2": "10000", "7/3": "10000", "7/4": "10000", "7/5": "10000", "7/6": "10000", "7/7": "10000", "7/8": "10000", "7/9": "10000", "7/10": "10000", "7/11": "10000", "7/12": "10000", "7/13": "10000", "7/14": "10000", "7/15": "10000", "7/16": "10000", "7/17": "10000", "7/18": "10000", "7/19": "10000", "7/20": "10000", "7/21": "10000", "7/22": "10000", "7/23": "10000", "7/24": "10000", "7/25": "10000", "7/26": "10000", "7/27": "10000", "7/28": "10000", "7/29": "10000"}   # "task/state": repeat ids (id 10001 also on 4 states)
OUT_NAME = "step17"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
STOP_AT_H = 4.0                    # no new work after this many hours of the session; running jobs are stopped
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 17 at {REPO_REF}: first-inference features, tasks {TASKS} x states 0-29, 124 episodes "
      f"(id 10000 on every state, id 10001 on 4); stop after {STOP_AT_H} h, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
previous = [p for p in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True))
            if os.path.isdir(p) and os.path.exists(f"{p}/repo_commit.txt")]
assert len(previous) <= 1, f"attach only the latest version's output, found {previous}"
for prev in previous:
    print("RESUMING from", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
if not previous:
    print("FRESH START: no earlier output attached. If this is the second session, stop and add it as input.", flush=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 15, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






SEARCH = f"cd {REPO} && {PY} -u noise_reachability/deep_noise_search.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS} --family repeat"
ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step17.py --root {OUT_ROOT}"


def feature_dir(task):
    return f"{OUT_ROOT}/task_{task}/features"


def n_ids(spec):
    return sum(int(p.split("-")[1]) - int(p.split("-")[0]) + 1 if "-" in p else 1 for p in spec.split(","))


def run_stage(name, jobs, total, max_parallel=4):
    """True when the stage has no work left: a job that failed stops the run, the deadline leaves work for a next version."""
    print(f"{name}: {len(jobs)} of {total} jobs to run; {hours_left():.1f} h left", flush=True)
    if not jobs:
        return True
    if hours_left() <= 0:
        return False
    codes = run_jobs(jobs, max_parallel=max_parallel)
    if failed(codes):
        fail(f"{name} failed: {failed(codes)}")
        return False
    return all(c == 0 for c in codes.values()) and len(codes) == len(jobs)

## 8. Wrapper check on each task's real model (a few minutes); stops the run if it fails

In [ ]:
CHECK_DONE = True
for t in TASKS:
    r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step5a.py --task_id {t} --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}",
           check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"check_step5a failed on task {t}")
        CHECK_DONE = False

## 9. Feature episodes: one job per map state (120 jobs), four at a time

Each job runs `deep_noise_search.py --family repeat --feature_dir ...` on one state; the features of the first
inference go to `task_<t>/features/json/`, the episode's row to `task_<t>/features/per_episode_repeat_state<i>.csv`.

In [ ]:
FEATURES_DONE = False
if CHECK_DONE and not FAILURE:
    jobs = {}
    for t in TASKS:
        out = feature_dir(t)
        for s in MAP_STATES:
            ids = IDS[f"{t}/{s}"]
            if len(read_csv(f"{out}/per_episode_repeat_state{s}.csv")) < n_ids(ids):
                jobs[f"features_task{t}_state{s}"] = (f"{SEARCH} --task_id {t} --init_state {s} --noise_ids {ids} "
                                                     f"--output_dir {out} --feature_dir {out}/json", len(jobs) % 2)
    FEATURES_DONE = run_stage("feature episodes", jobs, len(TASKS) * len(MAP_STATES))

## 10. Feature check, report and features file

The gate: at least 114 of 120 episodes reproduce step 16's committed rows, and the duplicate id's features are
identical on all 4 states. The test U1 itself (`step17_g1b.py`) runs locally on `step17_features.json`.

In [ ]:
GATE_DONE = False
if FEATURES_DONE and not FAILURE:
    if sh(f"{ANALYZE} --gate features", check=False, env=RUN_ENV).returncode != 0:
        fail("the feature check failed: investigate before reading anything")
    elif sh(f"{ANALYZE} --out {OUT_ROOT}/step17_report.txt --features {OUT_ROOT}/step17_features.json",
            check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")
    else:
        GATE_DONE = True
if FAILURE:
    print("FAILED:", FAILURE)
elif GATE_DONE:
    print("STEP 17 DONE: download step17_results.zip")
else:
    print("WORK REMAINS: add this version's output as input and Save & Run All again")

## 11. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step17_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step17_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)